In [1]:
from __future__ import annotations

import json
import pathlib
import random
from pathlib import Path

from kebab.utils.dataset.wikidata import wikidata_utils
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [2]:
target_entity_type = "Q5"

In [3]:
# pairs
dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_dataset.jsonl"
)

# pairs ground truth
ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_ground_truth.jsonl"
)

# Wikidata type hierarchy for filtering by type
type_hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2025-01-30"
    / "wikidata_type_hierarchy.jsonl"
)

In [4]:
# get the target entity type and its descendants
get_descendants = False

graph, type_id_to_node = wikidata_utils.load_type_hierarchy(type_hierarchy_path)
target_entity_types = (
    wikidata_utils.collect_all_subtypes(graph, type_id_to_node, target_entity_type)
    if get_descendants
    else {target_entity_type}
)
print(f"Target entity type: {target_entity_type} ({len(target_entity_types):,d} with descendants)")

Target entity type: Q5 (1 with descendants)


In [5]:
target_entity_types = {type_id_to_node[t]["name"] for t in target_entity_types}

In [6]:
# load the ground truth
with open(ground_truth_path, encoding="utf-8") as f:
    ground_truth = [json.loads(line) for line in f]

In [7]:
# load and filter fragments to only include the target entity types
fragments = {}
pairs = []
labels = []


def add_fragment(fragment: ResolvedWikidataEntity) -> ResolvedWikidataEntity:
    """Add the fragment to the set of known fragments."""
    if fragment.metadata["fragment_id"] not in fragments:
        fragments[fragment.metadata["fragment_id"]] = fragment
        fragment.evidence_map = None
        fragment.source_ids = None

    return fragments[fragment.metadata["fragment_id"]]


with open(dataset_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        left = add_fragment(ResolvedWikidataEntity.from_dict(d["left"]))
        right = add_fragment(ResolvedWikidataEntity.from_dict(d["right"]))

        if not set(left.wikidata_type).intersection(target_entity_types) or not set(right.wikidata_type).intersection(
            target_entity_types
        ):
            continue

        pairs.append((left, right))
        labels.append(ground_truth[i])

assert len(pairs) == len(labels)

print(f"Filtered to {len(pairs):,d} pairs where both entities are of the target types")
print(f"Positive pairs: {sum(labels):,d} ({sum(labels) / len(labels):.2%})")

Filtered to 831,431 pairs where both entities are of the target types
Positive pairs: 621,835 (74.79%)


In [8]:
target_count = 1_000

# sample pairs
sampled_pairs = []
sampled_labels = []
indices = list(range(len(pairs)))
random.shuffle(indices)

for i in indices[:target_count]:
    sampled_pairs.append(pairs[i])
    sampled_labels.append(labels[i])

print(f"Sampled {len(sampled_pairs):,d} pairs")
print(f"Positive pairs: {sum(sampled_labels):,d} ({sum(sampled_labels) / len(sampled_labels):.2%})")

Sampled 1,000 pairs
Positive pairs: 759 (75.90%)


In [9]:
# write the sampled pairs and labels to files
sampled_dataset_path = Path.cwd() / "sampled_rebel_linking_dataset.jsonl"

with open(sampled_dataset_path, "w", encoding="utf-8") as f:
    for pair in sampled_pairs:
        d = [
            pair[0].to_dict(minimal_repr=True),
            pair[1].to_dict(minimal_repr=True),
        ]
        f.write(json.dumps(d) + "\n")

sampled_ground_truth_path = Path.cwd() / "sampled_rebel_linking_ground_truth.jsonl"

with open(sampled_ground_truth_path, "w", encoding="utf-8") as f:
    for label in sampled_labels:
        f.write(json.dumps(label) + "\n")

In [10]:
# clustering dataset path
clustering_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_clustering_dataset.jsonl"
)

# clustering dataset ground truth
clustering_ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_clustering_ground_truth.jsonl"
)

In [11]:
# filter the clustering dataset and ground truth to only include the sampled entities
target_entity_count = 100
fragment_count = 0
entity_ids = set()
from collections import defaultdict
c = defaultdict(int)

with (open(clustering_dataset_path, encoding="utf-8") as f_ds,
    open(clustering_ground_truth_path, encoding="utf-8") as f_gt,
    open("sampled_rebel_clustering_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
    open("sampled_rebel_clustering_ground_truth.jsonl", "w", encoding="utf-8") as f_gt_out):
    for line_ds, line_gt in zip(f_ds, f_gt):
        fragment = json.loads(line_ds)
        entity_id = json.loads(line_gt)

        if entity_id not in entity_ids:
            if len(entity_ids) == target_entity_count:
                continue
            
            entity_ids.add(entity_id)
        
        fragment_count += 1
        f_ds_out.write(line_ds)
        f_gt_out.write(line_gt)
        c[entity_id] += 1

print(f"Filtered to {fragment_count:,d} fragments for {len(entity_ids):,d} entities")

Filtered to 8,574 fragments for 100 entities


In [12]:
for i, (k, v) in enumerate(sorted(c.items(), key=lambda x: -x[1])):
    print(f"{i}: {k}: {v}")

0: Q30: 2295
1: Q668: 711
2: Q16: 610
3: Q408: 504
4: Q1439: 457
5: Q20: 322
6: Q258: 299
7: Q843: 262
8: Q812: 244
9: Q801: 232
10: Q1408: 177
11: Q173: 168
12: Q1974: 147
13: Q664: 136
14: Q1904: 119
15: Q334: 111
16: Q1747689: 104
17: Q2707177: 98
18: Q49: 94
19: Q3125: 82
20: Q6465: 79
21: Q1726: 78
22: Q4478: 78
23: Q7325: 78
24: Q27407: 77
25: Q585: 70
26: Q1156: 61
27: Q1781: 56
28: Q7204: 51
29: Q133032: 51
30: Q253414: 43
31: Q157808: 38
32: Q8418: 35
33: Q164950: 34
34: Q51: 26
35: Q16562: 26
36: Q7203506: 25
37: Q777060: 24
38: Q12559: 24
39: Q11739: 21
40: Q5295: 21
41: Q185298: 19
42: Q5830907: 19
43: Q213608: 18
44: Q1595894: 17
45: Q184872: 15
46: Q180825: 15
47: Q3306996: 13
48: Q76615: 13
49: Q661049: 13
50: Q2038835: 12
51: Q6738184: 12
52: Q688333: 12
53: Q1301901: 11
54: Q656421: 11
55: Q233929: 11
56: Q1134743: 10
57: Q642682: 8
58: Q2315527: 8
59: Q805233: 8
60: Q133929: 7
61: Q1137675: 7
62: Q244679: 7
63: Q1725788: 7
64: Q921881: 7
65: Q949243: 6
66: Q792472: 6
